In [8]:
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from openai import AsyncOpenAI

from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory

from ragas.dataset_schema import SingleTurnSample

from ragas.metrics.collections import Faithfulness, AnswerRelevancy

from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall

C:\Users\Admin\AppData\Local\Temp\ipykernel_38636\3707136877.py:14: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall
C:\Users\Admin\AppData\Local\Temp\ipykernel_38636\3707136877.py:14: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall


In [9]:
import sys
import importlib.util

print("Python Executable:", sys.executable)

# This will tell us exactly which file Python is trying to load
spec = importlib.util.find_spec("ragas.embeddings.base")
if spec:
    print("Loading Ragas from:", spec.origin)
else:
    print("Ragas not found in path.")

Python Executable: c:\Users\Admin\Documents\MavenAIProject\.venv\Scripts\python.exe
Loading Ragas from: c:\Users\Admin\Documents\MavenAIProject\.venv\Lib\site-packages\ragas\embeddings\base.py


In [10]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [11]:
ls_client = Client()

In [12]:
dataset = ls_client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [13]:
dataset

Dataset(name='rag-evaluation-dataset', description='RAG evaluation dataset', data_type=<DataType.kv: 'kv'>, id=UUID('5c4f218d-ab09-4080-a2e8-decce8d14b8c'), created_at=datetime.datetime(2026, 10, 26, 2, 40, 54, 745597, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 10, 26, 2, 40, 54, 745597, tzinfo=TzInfo(0)), example_count=62, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'Windows-11-10.0.26200-SP0', 'sdk_version': '0.14.12', 'runtime_version': '3.12.11', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [14]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[25].outputs

{'ground_truth': 'The frame has a stated capacity of 220 pounds. It must be secured into the ground with the included stakes.',
 'reference_context_ids': ['B0C3VKW1DJ'],
 'reference_descriptions': ['SUPER DEAL 40" Kids Web Tree Swing Saucer Swing Set Combo - 40\'\' Purple Web Tree Swing + Heavy Duty A-Frame Metal Swing Set, Resilient and Resistant to All Types of Weather Combo Set - This product includes 1 x swing frame + 1 x 40” round saucer swing set. Ideal choice for your garden, backyard, playground and more outdoor places, suitable for a single adult or multiple kids/teens to swing; Create your own/ kid’s own swing area with our Outdoor Swing Combo! Swing Frame - Crafted from 2-inch heavy duty weather-resistant steel tubes and hardware with a powder-coated paint finish for superior durability; A-frame ensures extra sturbility ; Up to 220 lbs high weight capacity meets most of your needs and provides a safe stand for a kid- and even a couple of friends - to lay back, sway, have som

In [15]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[25].inputs

{'question': 'How much weight can the swing frame support, and does it need to be anchored?'}

In [16]:
reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[25].inputs
reference_output = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[25].outputs

In [17]:
qdrant_client = QdrantClient("url=http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="toys_and_games_small_sample",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["all_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, query):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{query}    
"""

    return prompt


def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    return response.choices[0].message.content


def rag_pipeline(query, qdrant_client, top_k=5):

    retrieved_context = retrieve_data(query, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    answer = generate_answer(prompt)

    final_answer = {
        "answer": answer,
        "question": query,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [18]:
rag_pipeline("Can I get a toy car?", qdrant_client)

{'answer': 'Yes. We have several toy cars available, including:\n- Disney Pixar Cars 3 diecast character vehicles (1:55 scale; each character vehicle is sold separately)\n- Hot Wheels Monster Truck 1:24 Scale 2022 Bone Shaker\n- Jada Toys Big Time Muscle Drift RC 1:10 Scale (2019 Ford Mustang, wide body, black)\n- Kidstinct RC Tank for Kids (1:16 scale, 2.4GHz, 4WD, with lights/music and turrets)',
 'question': 'Can I get a toy car?',
 'retrieved_context_ids': ['B0718T5BXP',
  'B08V616T5G',
  'B09G7K3JWQ',
  'B0BC8TJGQP',
  'B07JYJNTJ8'],
 'retrieved_context': ['Disney Car Toys 3 Diecast Junior Moon Vehicle New 1:55 scale character vehicles from Disney Pixar Cars3! Authentic true to movie decos and details that highlight each personality. Wide variety to create or enhance your collection. Eachdie-cast sold separately, subject to availability. Disney Pixar Cars 3 characters are rolling in! This exciting assortment of new 1:55 scale vehicles offers huge variety with true to movie details

In [19]:
reference_input

{'question': 'How much weight can the swing frame support, and does it need to be anchored?'}

In [20]:
reference_output

{'ground_truth': 'The frame has a stated capacity of 220 pounds. It must be secured into the ground with the included stakes.',
 'reference_context_ids': ['B0C3VKW1DJ'],
 'reference_descriptions': ['SUPER DEAL 40" Kids Web Tree Swing Saucer Swing Set Combo - 40\'\' Purple Web Tree Swing + Heavy Duty A-Frame Metal Swing Set, Resilient and Resistant to All Types of Weather Combo Set - This product includes 1 x swing frame + 1 x 40” round saucer swing set. Ideal choice for your garden, backyard, playground and more outdoor places, suitable for a single adult or multiple kids/teens to swing; Create your own/ kid’s own swing area with our Outdoor Swing Combo! Swing Frame - Crafted from 2-inch heavy duty weather-resistant steel tubes and hardware with a powder-coated paint finish for superior durability; A-frame ensures extra sturbility ; Up to 220 lbs high weight capacity meets most of your needs and provides a safe stand for a kid- and even a couple of friends - to lay back, sway, have som

In [21]:

result = rag_pipeline(reference_input["question"], qdrant_client)

In [22]:
result

{'answer': 'The swing frame is rated for a maximum weight capacity of 220 lb. Yes, it needs to be anchored: it’s required to be secured into the ground using the included stakes to ensure stability and safety.',
 'question': 'How much weight can the swing frame support, and does it need to be anchored?',
 'retrieved_context_ids': ['B0C3VKW1DJ',
  'B08GL635DH',
  'B00TSUE9F0',
  'B08PDQY3DP',
  'B07L249MRS'],
 'retrieved_context': ['SUPER DEAL 40" Kids Web Tree Swing Saucer Swing Set Combo - 40\'\' Purple Web Tree Swing + Heavy Duty A-Frame Metal Swing Set, Resilient and Resistant to All Types of Weather Combo Set - This product includes 1 x swing frame + 1 x 40” round saucer swing set. Ideal choice for your garden, backyard, playground and more outdoor places, suitable for a single adult or multiple kids/teens to swing; Create your own/ kid’s own swing area with our Outdoor Swing Combo! Swing Frame - Crafted from 2-inch heavy duty weather-resistant steel tubes and hardware with a powde

In [23]:
openai_client = AsyncOpenAI()

In [24]:
ragas_llm = llm_factory("gpt-6-luna", client=openai_client, max_completion_tokens=4000)

args = ragas_llm.model_args
args.pop("max_tokens", None)
args.pop("top_p", None)
args["temperature"] = 0.0
args["max_completion_tokens"] = 4000

ragas_embeddings = embedding_factory("openai", model="text-embedding-3-small", client=openai_client)

In [25]:
async def ragas_faitfulness(run, example):

    scorer = Faithfulness(llm=ragas_llm)

    return await scorer.ascore(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

In [26]:
await ragas_faitfulness(result, reference_output)

MetricResult(value=1.0)

In [27]:
async def ragas_relevancy(run, example):

    scorer = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.ascore(
        user_input=run["question"],
        response=run["answer"]
    )

In [28]:
await ragas_relevancy(result, reference_output)

MetricResult(value=0.8835410662047049)

In [29]:
async def ragas_context_precision_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [30]:
await ragas_context_precision_id_based(result, reference_output)

c:\Users\Admin\Documents\MavenAIProject\.venv\Lib\site-packages\langsmith\_internal\_addressing.py:104: LangSmithBetaWarning: Addressing runs with `langsmith.address` is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


0.2

In [31]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [32]:
await ragas_context_recall_id_based(result, reference_output)

1.0